## Before you run a single cell in this notebook - terminal pre-flight

Open a JupyterLab terminal (**File > New > Terminal**) and run this first:

```bash
cd /workspace/finpost
git pull --ff-only
git status
pip install -e ".[dev,dpo-reference]"
python -c "import finpost; print(finpost.__file__)"
python -c "import torch; print('cuda:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"
```

If `import finpost` fails, follow `docs/runbooks/runpod-bootstrap.md` and add `/workspace/finpost/src` to a `.pth` file. If CUDA is false after install, follow the torch downgrade note in that runbook.

This notebook assumes the SFT ablation has finished and that the `combined` final checkpoint is the chosen SFT model.

# Phase 1 RunPod DPO - Qwen 0.5B from combined SFT

This notebook is the Direct Preference Optimization operator surface for Phase 1. It starts from the final `combined` SFT checkpoint produced by `notebooks/sft_phase1_runpod_ablation_2000.ipynb`, builds a fixed offline preference dataset, runs a Qwen DPO canary, runs the full DPO baseline, converts the checkpoint to Hugging Face format, and evaluates Base vs SFT-best vs SFT+DPO.

**Important:** this notebook requires the DPO implementation workstream to be present. If `scripts/build_dpo_pairs.py` or `finpost.training.dpo_train` is missing, the setup cell below will stop immediately. That is intentional.

## Step 1 - Sanity-check the pod and repo

Expected hardware: one 48 GB GPU, ideally A40, RTX A6000, or RTX 6000 Ada. The older Quadro RTX 6000 is a 24 GB card and is not the default target for full DPO fine-tuning.

In [ ]:
import subprocess

subprocess.run(['nvidia-smi'], check=False)
subprocess.run(['df', '-h', '/workspace'], check=False)


In [ ]:
import json
import os
import sys
import warnings
from pathlib import Path

# Suppress a transformers false-positive triggered on Qwen tokenizers.
# The warning fires on any tokenizer whose pre-tokenization regex resembles
# a known Mistral bug pattern; Qwen is not affected and tokenization is correct.
warnings.filterwarnings("ignore", message=".*incorrect regex pattern.*")

import yaml

REPO_ROOT = Path('/workspace/finpost')
os.chdir(REPO_ROOT)
print('cwd:', Path.cwd())

subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'log', '--oneline', '-1'], check=False)

import torch
import finpost

print('finpost:', finpost.__version__)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

required_files = [
    Path('scripts/build_dpo_pairs.py'),
    Path('src/finpost/training/dpo.py'),
    Path('src/finpost/training/dpo_train.py'),
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        'DPO implementation files are not present yet. Pull the branch/commit that implements '
        f'.scratch/phase1-dpo-comparison before running this notebook. Missing: {missing}'
    )

print('DPO implementation files present.')


## Step 2 - Study constants

Chosen SFT checkpoint: `combined` at step **500**, pulled from
`sl891/finpost-phase1-sft-ablation`. This was determined empirically by the
Phase 1 SFT ablation — combined-500 achieved the highest GSM8K accuracy (0.276)
of any arm or step, and was the only checkpoint that generalized well across
both GSM8K and MATH. All three arms overfit past step 500; 2000 steps was
overkill. See `docs/phase1-sft-study.html` for the full findings.

If you want to use a different arm or step, update `SFT_ARM`,
`SFT_CHECKPOINT_STEP`, and `SFT_HF_SUBFOLDER` below.


In [ ]:
SFT_ARM = 'combined'
SFT_CHECKPOINT_STEP = 500          # step within the SFT run; empirically best on GSM8K + MATH
SFT_HF_REPO = 'sl891/finpost-phase1-sft-ablation'
SFT_HF_SUBFOLDER = f'{SFT_ARM}/step-{SFT_CHECKPOINT_STEP:04d}'  # combined/step-0500

# SFT_HF_CHECKPOINT is set to the downloaded local path by the fetch cell below.
# Do not edit this line; edit SFT_ARM / SFT_CHECKPOINT_STEP above instead.
SFT_HF_CHECKPOINT = None

PAIR_RUN_NAME = f'qwen_{SFT_ARM}_{SFT_CHECKPOINT_STEP}s_k8_v1'
PAIR_OUT_DIR = Path(f'results/dpo_pairs/{PAIR_RUN_NAME}')

DPO_RUN_NAME = f'qwen-{SFT_ARM}-step{SFT_CHECKPOINT_STEP}-dpo'
DPO_EXPERIMENT_DIR = Path('experiments/dpo')
DPO_YAML = DPO_EXPERIMENT_DIR / 'qwen_dpo_baseline.yaml'
DPO_CHECKPOINT_DIR = Path(f'results/checkpoints/{DPO_RUN_NAME}')
DPO_HF_DIR = Path(f'results/checkpoints/{DPO_RUN_NAME}-hf')

EVAL_OUT_DIR = Path('results/evals/dpo_phase1_run_1')
EVAL_RUN_NAME = EVAL_OUT_DIR.name

SOURCES = ['gsm8k', 'math']
HELDOUT_TRAIN_N = 2000
SAMPLES_PER_PROMPT = 8
TEMPERATURE = 0.8
MAX_NEW_TOKENS = 768
MAX_NEW_TOKENS_GSM8K = 256
MAX_NEW_TOKENS_MATH = 768
GENERATION_BATCH_SIZE = 64
SEED = 42

DTYPE = 'bfloat16'
MAX_SEQ_LEN = 1024
DPO_MAX_STEPS = 500
DPO_CANARY_STEPS = 50
DPO_LR = 5.0e-6
DPO_BETA = 0.1
PAIR_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
DATALOADER_NUM_WORKERS = 2
PIN_MEMORY = True
WARMUP_STEPS = 50
CHECKPOINT_EVERY_N_STEPS = 250
CHECKPOINT_RETENTION_LAST_N = 4

EVAL_N = 500
EVAL_SEED = 42
BATCH_SIZE_GSM8K = 128
BATCH_SIZE_MATH = 128
GPU_COST_PER_HOUR = 0.44

print('SFT HF repo:       ', SFT_HF_REPO)
print('SFT subfolder:     ', SFT_HF_SUBFOLDER)
print('pair out dir:      ', PAIR_OUT_DIR)
print('DPO run name:      ', DPO_RUN_NAME)
print('DPO yaml:          ', DPO_YAML)
print('DPO ckpt dir:      ', DPO_CHECKPOINT_DIR)
print('DPO HF dir:        ', DPO_HF_DIR)
print('eval out dir:      ', EVAL_OUT_DIR)
print('effective pair batch:', PAIR_BATCH_SIZE * GRAD_ACCUM_STEPS)


In [ ]:
# Download the chosen SFT checkpoint from HuggingFace Hub to the pod volume.
# snapshot_download is idempotent: if the files are already present it skips
# the network transfer. The write token in your pod env vars has read access
# too, so no re-authentication is needed.
#
# The repo mirrors the subfolder structure, so files land at:
#   _dl_root / combined / step-0500 / config.json
#   _dl_root / combined / step-0500 / model.safetensors
#   ...
# We then reassign SFT_HF_CHECKPOINT to that leaf directory.
from huggingface_hub import snapshot_download

_dl_root = Path('results/checkpoints/_sft_hf_cache')
print(f'Fetching {SFT_HF_REPO}/{SFT_HF_SUBFOLDER} ...')
snapshot_download(
    repo_id=SFT_HF_REPO,
    allow_patterns=f'{SFT_HF_SUBFOLDER}/**',
    local_dir=str(_dl_root),
    local_dir_use_symlinks=False,
)

SFT_HF_CHECKPOINT = _dl_root / SFT_HF_SUBFOLDER
print('SFT checkpoint ready at:', SFT_HF_CHECKPOINT)
for p in sorted(SFT_HF_CHECKPOINT.iterdir()):
    print(' ', p.name)


## Step 3 - Verify the chosen SFT checkpoint

The SFT notebook converts each trajectory checkpoint to Hugging Face format and deletes raw trainer checkpoints to save disk. DPO starts from the final converted `combined` checkpoint.

In [ ]:
required_sft_files = ['config.json', 'model.safetensors']
missing = [name for name in required_sft_files if not (SFT_HF_CHECKPOINT / name).exists()]
if missing:
    raise FileNotFoundError(
        f'Missing required files in {SFT_HF_CHECKPOINT}: {missing}. '
        'Run the SFT ablation notebook through the conversion cells first.'
    )

print('SFT checkpoint exists and has core HF files:')
for path in sorted(SFT_HF_CHECKPOINT.iterdir()):
    print(' ', path.name)


## Step 4 - Build fixed offline preference pairs

This samples `K=8` completions per held-out training prompt from the chosen SFT checkpoint, grades them with the exact-answer verifier path, and writes a frozen DPO pair dataset. Test prompts must never be used here.

In [ ]:
def run_and_tail(
    cmd: list[str],
    *,
    label: str,
    tail_lines: int = 40,
    env_overrides: dict[str, str] | None = None,
) -> None:
    """Run a subprocess with live output and keep a failure tail."""
    print(f'=== {label} ===')
    print(' '.join(str(part) for part in cmd))
    env = None
    if env_overrides:
        env = {**os.environ, **env_overrides}
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    tail: list[str] = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        tail.append(line.rstrip('\n'))
        if len(tail) > tail_lines:
            tail.pop(0)
    returncode = proc.wait()
    if returncode != 0:
        print('\n--- output tail ---')
        print('\n'.join(tail[-tail_lines:]))
        raise RuntimeError(f'{label} failed with exit code {returncode}')
    print(f'=== {label} passed ===')


In [ ]:
pair_cmd = [
    sys.executable,
    'scripts/build_dpo_pairs.py',
    '--sft-checkpoint', str(SFT_HF_CHECKPOINT),
    '--sources', *SOURCES,
    '--heldout-train-n', str(HELDOUT_TRAIN_N),
    '--samples-per-prompt', str(SAMPLES_PER_PROMPT),
    '--temperature', str(TEMPERATURE),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--max-new-tokens-gsm8k', str(MAX_NEW_TOKENS_GSM8K),
    '--max-new-tokens-math', str(MAX_NEW_TOKENS_MATH),
    '--generation-batch-size', str(GENERATION_BATCH_SIZE),
    '--seed', str(SEED),
    '--out-dir', str(PAIR_OUT_DIR),
    '--device', 'cuda',
]
run_and_tail(pair_cmd, label='build DPO preference pairs')


In [ ]:
manifest_path = PAIR_OUT_DIR / 'manifest.json'
pairs_path = PAIR_OUT_DIR / 'pairs.jsonl'
completions_path = PAIR_OUT_DIR / 'completions.jsonl'

for path in [manifest_path, pairs_path, completions_path]:
    if not path.exists():
        raise FileNotFoundError(f'Missing expected pair artifact: {path}')

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
pair_count = sum(1 for _ in pairs_path.open('r', encoding='utf-8'))
completion_count = sum(1 for _ in completions_path.open('r', encoding='utf-8'))

print('manifest:')
print(json.dumps(manifest, indent=2)[:4000])
print('\npairs:', pair_count)
print('completions:', completion_count)

if pair_count == 0:
    raise RuntimeError('No DPO pairs were produced. Stop and inspect all-correct/all-incorrect counts.')


## Step 5 - Write the DPO config

The policy and reference checkpoints are both initialized from the chosen SFT checkpoint. The reference model must stay frozen; only the policy model updates.

In [ ]:
dpo_config = {
    'model': {
        'policy_checkpoint': str(SFT_HF_CHECKPOINT),
        'reference_checkpoint': str(SFT_HF_CHECKPOINT),
        'dtype': DTYPE,
        'use_safetensors': True,
    },
    'data': {
        'pairs_path': str(pairs_path),
        'manifest_path': str(manifest_path),
        'tokenized_cache_path': str(PAIR_OUT_DIR / 'pairs.tokenized.pt'),
        'rebuild_tokenized_cache': False,
        'seed': SEED,
    },
    'dpo': {
        'beta': DPO_BETA,
    },
    'training': {
        'max_steps': DPO_MAX_STEPS,
        'warmup_steps': WARMUP_STEPS,
        'lr': DPO_LR,
        'weight_decay': 0.01,
        'grad_accum_steps': GRAD_ACCUM_STEPS,
        'grad_clip': 1.0,
        'checkpoint_every_n_steps': CHECKPOINT_EVERY_N_STEPS,
        'per_device_pair_batch_size': PAIR_BATCH_SIZE,
        'dataloader_num_workers': DATALOADER_NUM_WORKERS,
        'pin_memory': PIN_MEMORY,
    },
    'packing': {
        'max_seq_len': MAX_SEQ_LEN,
    },
    'logging': {
        'wandb_project': 'finpost-phase1-dpo-runpod',
        'run_name': DPO_RUN_NAME,
    },
    'checkpointing': {
        'save_dir': str(DPO_CHECKPOINT_DIR),
        'retention_last_n': CHECKPOINT_RETENTION_LAST_N,
        'resume_from': None,
    },
}

DPO_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
DPO_YAML.write_text(yaml.safe_dump(dpo_config, sort_keys=False), encoding='utf-8')
print(DPO_YAML.read_text(encoding='utf-8'))


## Step 6 - Qwen DPO canary

This is the hard stop/go gate before full DPO. If the canary fails, do not launch the full run. Fix the loss, batch size, pair format, or checkpoint paths first.

In [ ]:
# Canary is intentionally wandb-offline: 50 steps is a smoke test, not a run
# worth syncing. The env override is scoped to this subprocess only via
# run_and_tail's env_overrides arg, so the full run cell below still picks up
# WANDB_API_KEY from the pod env and syncs live.
canary_cmd = [
    sys.executable,
    '-m', 'finpost.training.dpo_train',
    '--config', str(DPO_YAML),
    '--device', 'cuda',
    '--max-steps', str(DPO_CANARY_STEPS),
]
run_and_tail(
    canary_cmd,
    label='Qwen DPO canary',
    env_overrides={'WANDB_MODE': 'offline'},
)


In [ ]:
subprocess.run(['df', '-h', '/workspace'], check=False)
subprocess.run(['nvidia-smi'], check=False)


### Drop canary checkpoints before the full run

The canary writes into the same ``DPO_CHECKPOINT_DIR`` as the full run. If we
do not clean up first, two things go wrong:

1. The canary's final checkpoint (~1-2 GB) sits on disk during the full run.
2. The ``dpo-find-checkpoint`` cell after training picks the highest-step
   directory under ``DPO_CHECKPOINT_DIR``. If the full run dies partway, that
   could be a stale canary step, and we would convert and evaluate the wrong
   model.

Dropping the canary checkpoints here makes the full run start from a clean
directory. The canary's logs in ``wandb/offline-run-*`` are tiny and left
alone for post-mortem if you ever need them.


In [ ]:
import shutil

if DPO_CHECKPOINT_DIR.exists():
    print(f'canary leftovers under {DPO_CHECKPOINT_DIR}:')
    for path in sorted(DPO_CHECKPOINT_DIR.iterdir()):
        print(' ', path.name)
    shutil.rmtree(DPO_CHECKPOINT_DIR)
    print(f'\nremoved {DPO_CHECKPOINT_DIR}')
else:
    print(f'{DPO_CHECKPOINT_DIR} does not exist yet - nothing to clean')

print('\ndisk after cleanup:')
subprocess.run(['df', '-h', '/workspace'], check=False)


## Step 7 - Full DPO run

Run this only after the canary exits cleanly. Keep the pod open until the final checkpoint has been converted and the eval artifacts have been packaged.

In [ ]:
full_dpo_cmd = [
    sys.executable,
    '-m', 'finpost.training.dpo_train',
    '--config', str(DPO_YAML),
    '--device', 'cuda',
]
run_and_tail(full_dpo_cmd, label='full DPO run', tail_lines=80)


## Step 8 - Verify and convert the final DPO checkpoint

The eval CLI expects Hugging Face format. Convert the final DPO trainer checkpoint after the full run completes.

In [ ]:
def checkpoint_step(path: Path) -> int:
    digits = ''.join(ch for ch in path.name if ch.isdigit())
    return int(digits) if digits else -1

step_dirs = sorted(DPO_CHECKPOINT_DIR.glob('step-*'), key=checkpoint_step)
if not step_dirs:
    raise FileNotFoundError(f'No DPO step-* checkpoints found under {DPO_CHECKPOINT_DIR}')

FINAL_DPO_STEP_DIR = step_dirs[-1]
print('final DPO checkpoint:', FINAL_DPO_STEP_DIR)
for path in sorted(FINAL_DPO_STEP_DIR.iterdir()):
    print(' ', path.name)


In [ ]:
convert_cmd = [
    sys.executable,
    'scripts/convert_checkpoint_to_hf.py',
    '--checkpoint-dir', str(FINAL_DPO_STEP_DIR),
    '--base-model-id', 'Qwen/Qwen2.5-0.5B',
    '--out-dir', str(DPO_HF_DIR),
    '--dtype', DTYPE,
]
run_and_tail(convert_cmd, label='convert DPO checkpoint to HF')

for name in ['config.json', 'model.safetensors']:
    if not (DPO_HF_DIR / name).exists():
        raise FileNotFoundError(f'Missing {name} in converted DPO HF dir: {DPO_HF_DIR}')
print('DPO HF checkpoint ready:', DPO_HF_DIR)


## Step 9 - Evaluate Base vs SFT-best vs SFT+DPO

Same eval seed and sample count as the SFT ablation. The goal is not just higher parse success; DPO needs to improve final-answer accuracy or be called inconclusive/harmful.

In [ ]:
eval_cmd = [
    sys.executable,
    '-m', 'finpost.evals.eval_exact',
    '--checkpoints',
    'base=Qwen/Qwen2.5-0.5B',
    f'sft_best={SFT_HF_CHECKPOINT}',
    f'dpo={DPO_HF_DIR}',
    '--sources', *SOURCES,
    '--n', str(EVAL_N),
    '--seed', str(EVAL_SEED),
    '--out-dir', str(EVAL_OUT_DIR),
    '--batch-size-gsm8k', str(BATCH_SIZE_GSM8K),
    '--batch-size-math', str(BATCH_SIZE_MATH),
    '--gpu-cost-per-hour', str(GPU_COST_PER_HOUR),
    '--device', 'cuda',
]
run_and_tail(eval_cmd, label='evaluate Base vs SFT vs DPO', tail_lines=80)


## Step 10 - Display headline numbers

In [ ]:
summary_csv = EVAL_OUT_DIR / 'accuracy_summary.csv'
cost_json = EVAL_OUT_DIR / 'cost_summary.json'

print('=== accuracy_summary.csv ===')
print(summary_csv.read_text(encoding='utf-8'))

print('\n=== cost_summary.json ===')
print(cost_json.read_text(encoding='utf-8'))

try:
    import pandas as pd
    df = pd.read_csv(summary_csv)
    print('\n=== Compact comparison ===')
    print(df[['checkpoint', 'source', 'accuracy', 'parse_success_rate']].to_string(index=False))
except ImportError:
    print('pandas unavailable; read the CSV above.')


## Step 11 - Package artifacts for download

Download this tarball before stopping or terminating the pod.

In [ ]:
package_path = Path('results') / f'{DPO_RUN_NAME}_artifacts.tar.gz'
package_cmd = [
    'tar', '-czf', str(package_path),
    str(PAIR_OUT_DIR),
    str(DPO_YAML),
    str(DPO_CHECKPOINT_DIR),
    str(DPO_HF_DIR),
    str(EVAL_OUT_DIR),
]
subprocess.run(package_cmd, check=True)
subprocess.run(['ls', '-lah', str(package_path)], check=False)
print('artifact bundle:', package_path)


## Step 12 - Optional HF upload

Only push the final DPO HF checkpoint if you want to re-evaluate later from another pod. Requires `hf auth login` with a write token.

In [ ]:
# Uncomment to push.
# repo_id = f'sl891/finpost-{DPO_RUN_NAME}'
# subprocess.run(['hf', 'upload', repo_id, str(DPO_HF_DIR)], check=True)


## Final - Stop the pod

Download the artifact tarball first. Then stop the pod if you may continue analysis soon, or terminate it after all pair data, checkpoints, eval outputs, and logs have been preserved.